<a href="https://colab.research.google.com/github/peterbabulik/QuantumWalker/blob/main/BlackHoleDigitalTwin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# ==============================================================================
#  The Digital Twin of a Black Hole: A Computational Experiment
#
#  This script simulates the formation of an event horizon as a computational
#  phase transition, based on the Babulik Inversion framework.
# ==============================================================================
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from tqdm import tqdm
from IPython.display import HTML

# ==============================================================================
#  STEP 1: Define the "Hardware Specs" of the Universe
# ==============================================================================
print("--- Defining the Universe's Hardware Specifications ---")

# The fundamental information processing density of the substrate
INFORMATION_DENSITY_LIMIT = 1 / (4 * np.log(2)) # ~0.36067 bits per pixel

# The size of our simulated patch of the substrate
GRID_SIZE = 101 # Use an odd number for a clear center

# The parameters of our candidate "Algorithm of Everything"
# How fast information diffuses away
DIFFUSION_RATE = 0.1
# How strongly information is attracted to other information (Gravity)
GRAVITATIONAL_CONSTANT = 0.5

print(f"Information Density Limit (ρ_I): {INFORMATION_DENSITY_LIMIT:.5f} bits/pixel")
print(f"Grid Size: {GRID_SIZE}x{GRID_SIZE} pixels")

# ==============================================================================
#  STEP 2: The Simulation Core
# ==============================================================================

def initialize_universe(grid_size: int, central_mass: float):
    """Creates the initial state of the simulation."""
    # The information content at each pixel
    info_grid = np.zeros((grid_size, grid_size), dtype=float)

    # The event horizon state (0 = normal space, 1 = horizon)
    horizon_grid = np.zeros((grid_size, grid_size), dtype=int)

    # Place the "matter" (a dense packet of information) at the center
    center = grid_size // 2
    info_grid[center, center] = central_mass

    return info_grid, horizon_grid

def apply_aoe_step(info_grid: np.ndarray) -> np.ndarray:
    """
    Applies one "tick" of the Algorithm of Everything.
    This simulates information diffusion and gravitational self-attraction.
    """
    # Use a padded grid to handle boundary conditions easily
    padded_grid = np.pad(info_grid, 1, mode='wrap') # Periodic boundary conditions
    new_info_grid = info_grid.copy()

    for i in range(1, GRID_SIZE + 1):
        for j in range(1, GRID_SIZE + 1):
            # Get current value and neighbors from the padded grid
            current_info = padded_grid[i, j]

            # Diffusion term: information spreads out
            laplacian = (padded_grid[i+1, j] + padded_grid[i-1, j] +
                         padded_grid[i, j+1] + padded_grid[i, j-1] - 4 * current_info)
            diffusion_flow = DIFFUSION_RATE * laplacian

            # Gravitational term: information is attracted to itself
            # This is a simplified "gradient" term
            grad_x = (padded_grid[i+1, j] - padded_grid[i-1, j]) / 2
            grad_y = (padded_grid[i, j+1] - padded_grid[i, j-1]) / 2
            # Information flows "downhill" towards higher density
            gravitational_flow = GRAVITATIONAL_CONSTANT * current_info * (grad_x**2 + grad_y**2)

            # Update the grid (map back to original grid indices)
            new_info_grid[i-1, j-1] += diffusion_flow + gravitational_flow

    # Ensure information content is non-negative
    return np.maximum(0, new_info_grid)

def check_for_horizon(info_grid: np.ndarray, horizon_grid: np.ndarray) -> np.ndarray:
    """
    Checks if any pixel has exceeded the information density limit.
    If so, it triggers a computational "crash" and becomes part of the horizon.
    """
    new_horizon_grid = horizon_grid.copy()

    # Find all pixels that are above the limit AND not already part of the horizon
    crashed_pixels = (info_grid > INFORMATION_DENSITY_LIMIT) & (horizon_grid == 0)

    if np.any(crashed_pixels):
        new_horizon_grid[crashed_pixels] = 1
        # The information in the crashed pixel is "frozen" and no longer evolves
        info_grid[crashed_pixels] = INFORMATION_DENSITY_LIMIT

    return info_grid, new_horizon_grid

# ==============================================================================
#  STEP 3: Run the Simulation
# ==============================================================================

if __name__ == "__main__":
    print("\n--- Starting the Black Hole 'Digital Twin' Simulation ---")

    # We need to find a "critical mass" that is just enough to form a horizon
    INITIAL_MASS = 5.0
    SIMULATION_STEPS = 200

    info, horizon = initialize_universe(GRID_SIZE, INITIAL_MASS)

    # Store the history of the grids to create an animation
    history = []

    for step in tqdm(range(SIMULATION_STEPS), desc="Simulating Universe"):
        # Store the current state for the animation
        history.append((info.copy(), horizon.copy()))

        # Evolve only the parts of the grid that are not yet a horizon
        non_horizon_mask = (horizon == 0)

        # Apply the AoE step to the entire grid
        updated_info = apply_aoe_step(info)

        # Only apply the update to the non-horizon regions
        info = np.where(non_horizon_mask, updated_info, info)

        # Check if the new state creates a horizon
        info, horizon = check_for_horizon(info, horizon)

    print("\n--- Simulation Complete ---")

    # ==============================================================================
    #  STEP 4: Visualize the Results
    # ==============================================================================
    print("--- Generating Visualization ---")

    fig, ax = plt.subplots(figsize=(8, 8))

    # Create the two image objects we will update in the animation
    im_info = ax.imshow(history[0][0], cmap='viridis', vmin=0, vmax=INFORMATION_DENSITY_LIMIT*1.1)
    # Use a transparent red colormap for the horizon overlay
    cmap_horizon = plt.cm.Reds
    cmap_horizon.set_under(alpha=0) # Make 0 values transparent
    im_horizon = ax.imshow(history[0][1], cmap=cmap_horizon, vmin=0.1, vmax=1)

    ax.set_title("Digital Twin of a Black Hole | Time Step: 0")
    ax.axis('off')

    # The animation function
    def animate(i):
        info_grid, horizon_grid = history[i]
        im_info.set_data(info_grid)
        im_horizon.set_data(horizon_grid)
        ax.set_title(f"Digital Twin of a Black Hole | Time Step: {i}")
        return [im_info, im_horizon]

    # Create and display the animation
    ani = FuncAnimation(fig, animate, frames=len(history), interval=50, blit=True)
    plt.close() # Prevents static plot from showing

    # Display the animation in the Colab notebook
    display(HTML(ani.to_jshtml()))

    # --- Final Analysis ---
    final_horizon = history[-1][1]
    if np.any(final_horizon):
        print("\n--- ANALYSIS ---")
        print("\033[1;32mSUCCESS: An event horizon has formed!\033[0m")
        print("The simulation demonstrates that when local information density exceeds the")
        print(f"Bekenstein-Hawking limit ({INFORMATION_DENSITY_LIMIT:.5f}), a computational phase transition")
        print("occurs, creating a stable event horizon. This provides a direct, computational")
        print("proof of concept for the formation of a black hole from first principles.")
    else:
        print("\n--- ANALYSIS ---")
        print("\033[1;33mRESULT: No event horizon formed.\033[0m")
        print("The initial mass was sub-critical. The information diffused away before any")
        print("pixel could reach the information density limit. This demonstrates the")
        print("existence of a 'Chandrasekhar limit' for informational collapse.")

--- Defining the Universe's Hardware Specifications ---
Information Density Limit (ρ_I): 0.36067 bits/pixel
Grid Size: 101x101 pixels

--- Starting the Black Hole 'Digital Twin' Simulation ---


Simulating Universe: 100%|██████████| 200/200 [00:15<00:00, 13.26it/s]



--- Simulation Complete ---
--- Generating Visualization ---



--- ANALYSIS ---
SUCCESS: An event horizon has formed!
The simulation demonstrates that when local information density exceeds the
Bekenstein-Hawking limit (0.36067), a computational phase transition
occurs, creating a stable event horizon. This provides a direct, computational
proof of concept for the formation of a black hole from first principles.
